# Making Learning Visible: A Tiny Gradient Descent Experiment

This notebook is intentionally small. I did **not** use a machine-learning framework because I wanted to see what happens during training before abstracting it away.

> **Understand first. Build second.**

We use one input, one weight, a fixed bias, one label, squared error, and Gradient Descent.

## 1. Initial values

The model is:

`prediction = input × weight + bias`

For this experiment, the correct relationship is `2 × weight = 4`, so we expect training to move the weight toward `2`. We do **not** give that value to the training algorithm.

In [ ]:
input_val = 2
weight = 0.5
bias = 0
label = 4
learning_rate = 0.1
epochs = 5

## 2. Prediction and Loss

Loss only needs the prediction and the label. It does not need to know how the prediction was produced.

In [ ]:
def prediction(input_val, weight, bias):
    return input_val * weight + bias

def loss(pred, label):
    return (pred - label) ** 2

pred = prediction(input_val, weight, bias)
initial_loss = loss(pred, label)
print(f"prediction = {pred}")
print(f"loss = {initial_loss}")

## 3. Gradient and one update

For squared error and this linear model, the gradient with respect to the weight is:

`2 × (prediction - label) × input`

Gradient Descent then uses that gradient and the Learning Rate to update the weight.

In [ ]:
def gradient_weight(input_val, pred, label):
    return 2 * (pred - label) * input_val

def gradient_descent(weight, gradient, learning_rate):
    return weight - learning_rate * gradient

grad_w = gradient_weight(input_val, pred, label)
new_weight = gradient_descent(weight, grad_w, learning_rate)
new_pred = prediction(input_val, new_weight, bias)
new_loss = loss(new_pred, label)

print(f"gradient = {grad_w}")
print(f"weight: {weight} -> {new_weight}")
print(f"loss: {initial_loss} -> {new_loss}")

### AHA

A single update moves the weight from `0.5` to `1.7` and reduces the Loss from `9` to `0.36`.

The next important step is to **keep the updated weight** and use it for the next prediction.

## 4. Training loop

The key idea that finally clicked:

`current weight → prediction → Loss → gradient → Gradient Descent → updated weight → next epoch`

The updated parameter must survive from one epoch to the next.

In [ ]:
def training(initial_weight, epochs):
    current_weight = initial_weight
    history = []

    for epoch in range(epochs):
        pred = prediction(input_val, current_weight, bias)
        current_loss = loss(pred, label)
        history.append((epoch, current_weight, pred, current_loss))

        print(
            f"Epoch {epoch} | prediction = {pred:.4f} | "
            f"loss = {current_loss:.6f} | weight = {current_weight:.4f}"
        )

        grad_w = gradient_weight(input_val, pred, label)
        current_weight = gradient_descent(
            current_weight, grad_w, learning_rate
        )

    return current_weight, history

optimized_weight, history = training(weight, epochs)
print(f"Optimized weight = {optimized_weight:.5f}")

## 5. What went wrong while I was building it

These mistakes were more useful than a perfect first implementation:

- I tried to stop training when `prediction != label`. That stopped the model exactly when it needed to learn.
- I reset the weight inside the loop. That discarded what the previous epoch had learned.
- I used the old weight for prediction and the new weight for Loss. That mixed two model states in the same epoch.
- I initially thought about Loss as a universal threshold for a correct prediction. It is instead the objective being minimized.
- After the loop worked, I noticed that prediction was calculated twice. Refactoring `loss(pred, label)` made its responsibility clearer and removed the redundant calculation.

## 6. What I learned

The model was never given `weight = 2`.

It started at `0.5` and moved toward `2` because each update used information from the current error.

For this experiment, **learning** became concrete:

> The model adjusts a parameter based on its error so that future predictions produce a smaller Loss.

No PyTorch. No TensorFlow. No `model.fit(...)` and staring at it like a refrigerator. 🧊

The refrigerator can wait.